# LDA - Master Notebook.ipynb [DM]

<div class="alert alert-block alert-info">
Notebook Start

### LDA Topic Model - Master Notebook
#### ***Exploring GFC (2007-2009) and Covid-19 (2019-2022) datasets for:***
##### Inflation Report (IR) & Monetary Policy Report (MPR)
##### Financial Stability Report (FSR)
##### Monetary Policy Committee - Meeting Minutes

#### Installing and importing packages/libraries

In [ ]:
# !pip install PyPDF2 gensim nltk scikit-learn
# !pip install pdfplumber
# pip install pyLDAvis
# pip install pycryptodome

In [6]:
import os
import re
import nltk
from sklearn.feature_extraction.text import CountVectorizer
from gensim import corpora, models
from nltk.corpus import stopwords
from pathlib import Path
from collections import Counter

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Medin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Defining Functions

#### Extract text from PDFs function
PDF Plumber appears to be the best library to extract text. Other libraries like Fitz (MyMuPDF) and PyPDF would return words like "nance", "nancial", "signi", "ned", etc.


In [9]:
# Function to extract text using PDF Plumber
import pdfplumber

def extract_text_from_pdf(pdf_path):
    text = ""
    # Open the PDF with pdfplumber
    with pdfplumber.open(pdf_path) as pdf:
        # Iterate through pages
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:   # avoid None
                text += page_text + " "
    return text




#### Cleaning function
Stop words list had to be extended to capture "noise" words. Need to decide with the team if finance/policy filler words need to be added to the stop words list.

##### Cleaning Function Summary
This function prepares raw PDF‑extracted text for NLP tasks (e.g., topic modeling):

1. **Normalize case** → converts all text to lowercase.  
2. **Fix artifacts** → removes hyphenated line breaks, normalizes whitespace, and merges “non‑” prefixes.  
3. **Remove non‑alphabetic characters** → strips out numbers, punctuation, and symbols.  
4. **Tokenize** → splits text into individual words.  
5. **POS tagging + lemmatization** → uses NLTK’s WordNet lemmatizer with part‑of‑speech tags to reduce words to their dictionary form (e.g., *running → run*, *policies → policy*).  
6. **Filter noise** → removes stopwords and very short tokens (≤ 2 characters).  

Output: a clean list of meaningful, normalized tokens ready for topic modeling or sentiment analysis.


In [11]:
# Create stop words before cleaning datasets
# Base stopwords as a set
stop_words = set(stopwords.words('english'))

# Domain-specific stopwords
domain_stopwords = {
    "could", "would", "should", "might", "may", "must",
    "also", "since", "however", "therefore", "thus", "within", "among", "across",
    "figure", "table", "report", "section", "appendix", "percent", "number",
    "january", "february", "march", "april", "may", "june",
    "july", "august", "september", "october", "november", "december", "around", "year", "end", "bank", "england",
    "financial", "economic", "finance", "percentage"
}   

# Merge sets

# Update stop words to include domain stopwords and reduce "noise" words.
stop_words.update(domain_stopwords)

In [12]:
# New Cleaning Function 
# Download the POS tagger and WordNet resources
import nltk
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')   # optional, improves lemmatization coverage

from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet


# Initialize lemmatiser
lemmatiser = WordNetLemmatizer()

# Helper: map NLTK POS tags to WordNet POS tags
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # default

def clean_text(text):
    # 1. Normalize case
    text = text.lower()
    
    # 2. Fix hyphenated line breaks and normalize whitespace
    text = re.sub(r'-\s*\n', '', text)   # remove hyphen + newline
    text = re.sub(r'\n', ' ', text)      # replace newlines with space
    text = re.sub(r'\bnon[\s-]+', 'non', text)  # merge "non " or "non-" into "non"
    
    # 3. Remove non-alphabetic characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    
    # 4. Tokenize
    tokens = text.split()
    
    # 5. POS tagging + lemmatization
    pos_tags = pos_tag(tokens)
    
   
    lemmatised = []
    for word, tag in pos_tags:
        lemma = lemmatiser.lemmatize(word, get_wordnet_pos(tag))
        if lemma not in stop_words and len(lemma) > 2:
            lemmatised.append(lemma)

    return lemmatised


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Medin\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Medin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Medin\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


### Specify datasets, extract text from PDFs and build dictionary and corpus for each dataset. 

In [14]:
# Specifying datasets
datasets = {
    "FSR_Covid": "FSR - Covid",
    "FSR_GFC": "FSR - GFC",
    "IR_MPR_Covid": "IR - MPR - Covid",
    "IR_GFC": "IR - GFC",
    "MPC_Meeting_Covid": "MPC Meeting - Covid",
    "MPC_Meeting_GFC": "MPC Meeting - GFC"
}

# Extract, clean and build dictionary/corpus
from gensim import corpora

all_cleaned = {}
all_dicts = {}
all_corpora = {}

for name, folder_path in datasets.items():
    documents = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)
            text = extract_text_from_pdf(pdf_path)
            documents.append(text)
    
    # Clean documents
    cleaned_docs = [clean_text(doc) for doc in documents]
    all_cleaned[name] = cleaned_docs
    
    # Create dictionary & corpus
    dictionary = corpora.Dictionary(cleaned_docs)
    dictionary.filter_extremes(no_below=2, no_above=0.9)  # optional filtering
    corpus = [dictionary.doc2bow(doc) for doc in cleaned_docs]
    
    all_dicts[name] = dictionary
    all_corpora[name] = corpus
    
    print(f"{name}: Dictionary size {len(dictionary)}, Corpus size {len(corpus)}")


FSR_Covid: Dictionary size 2583, Corpus size 9


Cannot set gray non-stroke color because /'P1' is an invalid float value
Cannot set gray non-stroke color because /'P1' is an invalid float value


FSR_GFC: Dictionary size 2291, Corpus size 6


Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBB

IR_MPR_Covid: Dictionary size 2512, Corpus size 17
IR_GFC: Dictionary size 1888, Corpus size 12
MPC_Meeting_Covid: Dictionary size 1786, Corpus size 26
MPC_Meeting_GFC: Dictionary size 1789, Corpus size 36


### Determine best number of topics per dataset, run LDA and export HMTL visualisation and CSV files

##### How the optimal number of topics is selected

##### The workflow trains multiple LDA models with varying numbers of topics (from `start` to `limit` in steps).  For each model, a **coherence score** (`c_v`) is calculated, which measures how interpretable and semantically consistent the topics are with respect to the input texts. The number of topics that yields the **highest coherence score** is chosen as the optimal value.  
 
##### - Iterate over candidate topic counts  
##### - Compute coherence for each model  
##### - Select the topic count with the maximum coherence score


In [16]:
# Define function to determine best number of topics per dataset
from gensim.models import LdaModel, CoherenceModel

def compute_coherence_values(dictionary, corpus, texts, start=2, limit=10, step=1):
    coherence_values = []
    model_list = []
    for num_topics in range(start, limit, step):
        model = LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=num_topics,
            random_state=42,
            passes=10,
            alpha='auto',
            per_word_topics=True
        )
        model_list.append(model)
        coherencemodel = CoherenceModel(
            model=model,
            texts=texts,
            dictionary=dictionary,
            coherence='c_v'
        )
        coherence_values.append(coherencemodel.get_coherence())
    return model_list, coherence_values

# Function to pick the best number of topics
def get_optimal_num_topics(dictionary, corpus, texts, start=2, limit=15, step=1):
    model_list, coherence_values = compute_coherence_values(
        dictionary=dictionary,
        corpus=corpus,
        texts=texts,
        start=start,
        limit=limit,
        step=step
    )
    x = list(range(start, limit, step))
    best_index = coherence_values.index(max(coherence_values))
    best_num_topics = x[best_index]
    return best_num_topics, coherence_values

In [17]:
# Importing libraries for visualisation, LDA model and pandas.
import pyLDAvis
import pyLDAvis.gensim
import pandas as pd
from gensim.models import LdaModel

# Determine the best number of models, then run LDA topic model on each dataset --> Export HTML visualisation and CSV files for each dataset.
lda_models = {}

for name in datasets.keys():
    dictionary = all_dicts[name]
    corpus = all_corpora[name]
    texts = all_cleaned[name]
    
    # Find optimal number of topics
    best_num_topics, coherence_values = get_optimal_num_topics(
        dictionary, corpus, texts, start=2, limit=15, step=1
    )
    
    print(f"{name}: Optimal number of topics = {best_num_topics}")
    
    # Train LDA model
    lda_model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=best_num_topics,
        passes=15,
        random_state=42
    )
    lda_models[name] = lda_model
    
    # --- HTML Visualization ---
    lda_vis = pyLDAvis.gensim.prepare(lda_model, corpus, dictionary)
    html_filename = f"{name}_LDA_Topics_Visualisation.html"
    pyLDAvis.save_html(lda_vis, html_filename)
    print(f"{name}: LDA visualization saved as {html_filename}")
    
    # --- CSV Export ---
    topic_data = []
    for topic_id in range(best_num_topics):
        terms = lda_model.show_topic(topic_id, topn=20)  # top 20 words per topic
        for rank, (word, weight) in enumerate(terms, start=1):
            topic_data.append({
                "Dataset": name,
                "Topic": topic_id,
                "Rank": rank,
                "Word": word,
                "Weight": weight
            })
    
    df_topics = pd.DataFrame(topic_data)
    csv_filename = f"{name}_LDA_topics.csv"
    df_topics.to_csv(csv_filename, index=False)
    print(f"{name}: LDA topics exported to {csv_filename}")


FSR_Covid: Optimal number of topics = 8
FSR_Covid: LDA visualization saved as FSR_Covid_LDA_Topics_Visualisation.html
FSR_Covid: LDA topics exported to FSR_Covid_LDA_topics.csv
FSR_GFC: Optimal number of topics = 2
FSR_GFC: LDA visualization saved as FSR_GFC_LDA_Topics_Visualisation.html
FSR_GFC: LDA topics exported to FSR_GFC_LDA_topics.csv
IR_MPR_Covid: Optimal number of topics = 4
IR_MPR_Covid: LDA visualization saved as IR_MPR_Covid_LDA_Topics_Visualisation.html
IR_MPR_Covid: LDA topics exported to IR_MPR_Covid_LDA_topics.csv
IR_GFC: Optimal number of topics = 7
IR_GFC: LDA visualization saved as IR_GFC_LDA_Topics_Visualisation.html
IR_GFC: LDA topics exported to IR_GFC_LDA_topics.csv
MPC_Meeting_Covid: Optimal number of topics = 3
MPC_Meeting_Covid: LDA visualization saved as MPC_Meeting_Covid_LDA_Topics_Visualisation.html
MPC_Meeting_Covid: LDA topics exported to MPC_Meeting_Covid_LDA_topics.csv
MPC_Meeting_GFC: Optimal number of topics = 2
MPC_Meeting_GFC: LDA visualization save

#### Export terms saliency on each dataset

Term → the word itself

Freq → frequency in the corpus

Total → total occurrences

loglift → distinctiveness of the term

logprob → probability of the term in topics

Category → whether the term is “Default” or belongs to a specific topic

The formula to calculate "saliency" combines term frequency and distinctiveness (lift) to rank saliency --> Saliency(w) = Freq(w) × Distinctiveness(w)


In [19]:
# Import gensin_models
import pyLDAvis.gensim_models

# Calculate and export top salient terms for each dataset
for name, lda_model in lda_models.items():
    # Prepare pyLDAvis visualization for this dataset
    lda_vis = pyLDAvis.gensim_models.prepare(lda_model, all_corpora[name], all_dicts[name])

    # Copy the topic_info DataFrame
    term_data = lda_vis.topic_info.copy()

    # Compute saliency manually (Freq × loglift)
    term_data['saliency'] = term_data['Freq'] * term_data['loglift']

    # Sort by saliency and take top 30
    top_salient = term_data.sort_values(by='saliency', ascending=False).head(30)

    # Export to CSV
    csv_filename = f"{name}_LDA_salient_terms.csv"
    top_salient.to_csv(csv_filename, index=False)

    print(f"{name}: Top 30 salient terms exported to {csv_filename}")


FSR_Covid: Top 30 salient terms exported to FSR_Covid_LDA_salient_terms.csv
FSR_GFC: Top 30 salient terms exported to FSR_GFC_LDA_salient_terms.csv
IR_MPR_Covid: Top 30 salient terms exported to IR_MPR_Covid_LDA_salient_terms.csv
IR_GFC: Top 30 salient terms exported to IR_GFC_LDA_salient_terms.csv
MPC_Meeting_Covid: Top 30 salient terms exported to MPC_Meeting_Covid_LDA_salient_terms.csv
MPC_Meeting_GFC: Top 30 salient terms exported to MPC_Meeting_GFC_LDA_salient_terms.csv


<div class="alert alert-block alert-info">
Notebook End